In [86]:
pip install pycldf pandas

In [87]:
import pandas as pd
from pycldf import Wordlist

df = pd.read_csv("protoforms_final_with_sources.tsv", sep="\t")

In [88]:
# remove missing-form placeholders
df["FORM"] = df["FORM"].replace("nan", pd.NA)
df["TOKENS"] = df["TOKENS"].replace("n a n", pd.NA)

df = df[
    df["FORM"].notna()
]

df = df[
    df["FORM"].astype(str).str.strip() != ""
]

In [92]:
languages = (
    df[
        [
            "DOCULECT",
            "LANGUAGE_NAME",
            "FAMILY",
            "LATITUDE",
            "LONGITUDE"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

# Create UNIQUE language IDs
languages["ID"] = [
    f"l{i}"
    for i in range(1, len(languages) + 1)
]

languages = languages.rename(columns={
    "DOCULECT": "Glottocode",
    "LANGUAGE_NAME": "Name",
    "FAMILY": "Family",
    "LATITUDE": "Latitude",
    "LONGITUDE": "Longitude"
})

# Mapping DOCULECT -> unique ID
language_map = dict(
    zip(languages["Glottocode"], languages["ID"])
)

languages = languages[
    [
        "ID",
        "Name",
        "Glottocode",
        "Family",
        "Latitude",
        "Longitude"
    ]
]

# Clean numeric coordinate columns only
languages["Latitude"] = (
    languages["Latitude"]
    .replace("nan", pd.NA)
)

languages["Longitude"] = (
    languages["Longitude"]
    .replace("nan", pd.NA)
)

# Ensure numeric columns are truly numeric
languages["Latitude"] = pd.to_numeric(
    languages["Latitude"],
    errors="coerce"
)

languages["Longitude"] = pd.to_numeric(
    languages["Longitude"],
    errors="coerce"
)

languages.to_csv("languages.csv", index=False)

In [93]:
parameters = (
    df[
        [
            "CONCEPT",
            "CONCEPTICON_ID",
            "ORIGINAL_GLOSS"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
    .copy()
)

# Create UNIQUE parameter IDs
parameters["ID"] = [
    f"p{i}"
    for i in range(1, len(parameters) + 1)
]

parameters = parameters.rename(columns={
    "CONCEPT": "Name",
    "CONCEPTICON_ID": "Concepticon_ID",
    "ORIGINAL_GLOSS": "Original_Gloss"
})

# Mapping concept label -> parameter ID
concept_map = dict(
    zip(parameters["Name"], parameters["ID"])
)

parameters = parameters[
    [
        "ID",
        "Name",
        "Concepticon_ID",
        "Original_Gloss"
    ]
]

parameters = parameters.replace("nan", pd.NA)

parameters["Concepticon_ID"] = pd.to_numeric(
    parameters["Concepticon_ID"],
    errors="coerce"
)

parameters.to_csv("parameters.csv", index=False)


In [94]:
form_df = df.copy()

# Map concepts and languages to unique IDs
form_df["Parameter_ID"] = (
    form_df["CONCEPT"].map(concept_map)
)

form_df["Language_ID"] = (
    form_df["DOCULECT"].map(language_map)
)

# Unique form IDs
form_df["ID"] = [
    f"f{i}"
    for i in range(1, len(form_df) + 1)
]

forms = form_df.rename(columns={
    "FORM": "Form",
    "TOKENS": "Segments",
    "Source": "Source"
})

forms = forms[
    [
        "ID",
        "Language_ID",
        "Parameter_ID",
        "Form",
        "Segments",
        "Source"
    ]
]

forms["Source"] = (
    forms["Source"]
    .fillna("")
    .astype(str)
)

forms = forms.replace("nan", pd.NA)

forms.to_csv("forms.csv", index=False)

In [95]:
forms_records = pd.read_csv("forms.csv").to_dict("records")
languages_records = pd.read_csv("languages.csv").to_dict("records")
parameters_records = pd.read_csv("parameters.csv").to_dict("records")

ds = Wordlist.in_dir(".")

ds.add_component("LanguageTable")
ds.add_component("ParameterTable")

ds.add_columns(
    "LanguageTable",
    {
        "name": "Family",
        "datatype": "string"
    }
)

ds.write(
    FormTable=forms_records,
    LanguageTable=languages_records,
    ParameterTable=parameters_records
)

print("CLDF dataset written successfully.")

CLDF dataset written successfully.
